#### import deps

In [43]:
import os
import openai
from qdrant_client import QdrantClient
from langsmith import Client

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper


In [44]:
from typing import Annotated, List, Any
from operator import add

from IPython.display import Image, display
from jinja2 import Template
from pydantic import BaseModel, Field
import cohere
import instructor
import openai

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, convert_to_openai_messages
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langsmith import traceable, get_current_run_tree
from qdrant_client import QdrantClient, models
from qdrant_client.models import Document, FieldCondition, Filter, FusionQuery, MatchAny, Prefetch
from langgraph.checkpoint.postgres import PostgresSaver
from langchain_core.messages import ToolMessage
import uuid

from utils.tools import get_formatted_item_context, get_formatted_reviews_context, add_to_shopping_cart, get_shopping_cart, remove_from_cart, check_warehouse_availability, reserve_warehouse_items

### Download example reference data point from langsmith

In [45]:
from dotenv import load_dotenv #import 
import os 

load_dotenv("../../.env")
ls_client = Client()

In [46]:
dataset = ls_client.read_dataset(
    dataset_name="coordinator-delegation-evaluation"
)
print(dataset)

name='coordinator-delegation-evaluation' description='' data_type=<DataType.kv: 'kv'> id=UUID('3bce65b2-a574-4c1d-a848-5fb0b51c8376') created_at=datetime.datetime(2026, 8, 9, 21, 10, 41, 355957, tzinfo=TzInfo(0)) modified_at=datetime.datetime(2026, 8, 9, 21, 10, 41, 355957, tzinfo=TzInfo(0)) example_count=3 session_count=5 last_session_start_time=datetime.datetime(2026, 8, 10, 3, 50, 33, 408315) inputs_schema=None outputs_schema=None transformations=None metadata=None


In [47]:
reference_inputs = [item.inputs["input"] for item in list(ls_client.list_examples(dataset_name="coordinator-delegation-evaluation"))]


In [48]:
reference_inputs

[{'messages': [{'type': 'human',
    'content': 'Can I get some earphones and a laptop? Could you also fetch some user reviews about durability of each item suggested?',
    'additional_kwargs': {},
    'response_metadata': {}},
   {'id': 'resp_02be47646ce6cc93006a794b590edc81989ef02c24523fccc9',
    'type': 'ai',
    'content': [{'id': 'rs_02be47646ce6cc93006a794b597a6881988e8c926cf5c53cb5',
      'type': 'reasoning',
      'content': [],
      'summary': [],
      'encrypted_content': 'gAAAAABqeUta2OnVnglACzopFcZ3meM_uO7boTNqBsC3mDHleMIl4wpyx17o-zPBtUgCNBHRYu3P1on-7RoCEQ01t7GAorm2nXuwfAxb2VNoQIffmyx3-b7iP7ZXXwyGULouvkfO8jH-C4OBvftkVFVYpgaUU6gpU1Bnrm7CEBzoxO-fdpcUJzyROR602n1j-oUEl_6gsElLwzakzEPSKWu7SBRsJdSA3LZhs6WBn02eTL1S7AiEe6pO4cGJIZUDYAE3d13prcZxxg2GcsSaRhBfHRNJDChveuMUHywFJ63UUlcvzpYybttUMaV4jyhOupDGTiPNzg9TIHDygxBW-aJmiYCs_5RIZT1U4FruLJf7uSWOulRZ-0vVnzRKc0nnTUFUoKTaNa_w-Kph2t7PT6U_Y3vKOfFZ4LUMFKaLiQi0MhND-zQNKBq1Sy1sjlkmXvBnMKGM8o_nGLshvDuWwmxcT5UYx_f8KRjugpIyRZwLolLMQrR9KL9qwAE

In [49]:
reference_outputs = [item.outputs for item in list(ls_client.list_examples(dataset_name="coordinator-delegation-evaluation"))]

In [50]:
reference_outputs

[{'coordinator_agent': {'plan': [{'task': 'Reserve 1 unit each of product B0BRTRHQKX (Lekaby Wireless Earbuds Bluetooth 5.3) and B09XJJNCXZ (HP Newest Notebook Laptop, 14" HD Touchscreen) in the warehouse for the user\'s cart/order.',
     'agent': 'warehouse_manager_agent'}],
   'next_agent': 'warehouse_manager_agent',
   'final_answer': False}},
 {'coordinator_agent': {'plan': [{'task': 'Reserve the best-reviewed earphone and laptop from warehouse inventory after the cart is updated.',
     'agent': 'warehouse_manager_agent'}],
   'next_agent': 'warehouse_manager_agent',
   'final_answer': False}},
 {'coordinator_agent': {'plan': [],
   'iteration': 2,
   'next_agent': '',
   'final_answer': True}}]

### RAG pipeline 

### coordinator agent evaluation

In [60]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to invoke, either 'product_qna_agent', 'shopping_cart_agent', or 'warehouse_manager_agent'")
    task: str = Field(description="The task to be performed by the agent")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke, either 'product_qna_agent', 'shopping_cart_agent', or 'warehouse_manager_agent'")
    plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence")
    
    
class AgentProperties(BaseModel):
    final_answer: bool = False
    iteration: int = 0
    
class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False 
    plan: List[Delegation] = []
    next_agent: str = ""

class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    user_intent: str = ""
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""
    
class FinalAgentResponse(BaseModel):
    answer: str = Field(description="Answer to the question")


In [67]:

traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini",
    }
)
def coordinator_agent(state) -> dict: # agent node always gets state object
    prompt_template = """
        Your role is to delegate work to worker agents in order to solve user queries.
            - You should output the next agent to invoke.
            - Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the 'Plan' tool.
            - If you do not need to delegate to any agent, you should output a final answer via the 'FinalAgentResponse' tool.
            - Do not route to any agent if the user's query needs clarification or is irrelevant. Do it yourself.
            - Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
            - Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
            - Do not delegate work to the same agent twice in a row.

            ## Routing rules

            Match the user's latest message to one of these patterns:

            1. PRODUCT QUESTION - asks about products, specs, reviews, availability, suggestions.
            Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some options".
            -> Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

            2. EXPLICIT CART INSTRUCTION - uses cart vocabulary.
            Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
            -> Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

            3. EXPLICIT RESERVATION INSTRUCTION - uses warehouse/reservation vocabulary.
            Examples: "reserve these", "hold these for me", "check warehouse stock for X".
            -> Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

            Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents.
            "I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add to my cart" or "reserve" to trigger cart or warehouse routing.

            ## Available Agents

            - product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews, etc.
            - shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current state of the shopping cart.
            - warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of items in the warehouses.
            
    """
    
    template = Template(prompt_template)
    prompt = template.render()
    llm = ChatOpenAI(
        model="gpt-5.4-mini", 
        reasoning_effort = "low",
        use_responses_api = True
    )

    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke([
        SystemMessage(content=prompt),
        *state.messages
    ])
    
    final_answer = False
    answer = ""
    plan = []
    next_agent = ""
    
    def sanitise_response(response):
        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            plan = response.tool_calls[0].get("args").get("plan")
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None  # don't add a tool-call message to history when delegating
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")

            response = sanitise_response(response)


    return { 
        "messages": [response] if response is not None else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }



In [68]:
print(reference_inputs[0]["messages"])
output = coordinator_agent(
    State(
        messages=reference_inputs[0]["messages"],
        coordinator_agent=CoordinatorAgentProperties(
            iteration=0,
            final_answer=False,
            plan=[],
            next_agent=""
        ),
        answer=""
    )
)
output

[{'type': 'human', 'content': 'Can I get some earphones and a laptop? Could you also fetch some user reviews about durability of each item suggested?', 'additional_kwargs': {}, 'response_metadata': {}}, {'id': 'resp_02be47646ce6cc93006a794b590edc81989ef02c24523fccc9', 'type': 'ai', 'content': [{'id': 'rs_02be47646ce6cc93006a794b597a6881988e8c926cf5c53cb5', 'type': 'reasoning', 'content': [], 'summary': [], 'encrypted_content': 'gAAAAABqeUta2OnVnglACzopFcZ3meM_uO7boTNqBsC3mDHleMIl4wpyx17o-zPBtUgCNBHRYu3P1on-7RoCEQ01t7GAorm2nXuwfAxb2VNoQIffmyx3-b7iP7ZXXwyGULouvkfO8jH-C4OBvftkVFVYpgaUU6gpU1Bnrm7CEBzoxO-fdpcUJzyROR602n1j-oUEl_6gsElLwzakzEPSKWu7SBRsJdSA3LZhs6WBn02eTL1S7AiEe6pO4cGJIZUDYAE3d13prcZxxg2GcsSaRhBfHRNJDChveuMUHywFJ63UUlcvzpYybttUMaV4jyhOupDGTiPNzg9TIHDygxBW-aJmiYCs_5RIZT1U4FruLJf7uSWOulRZ-0vVnzRKc0nnTUFUoKTaNa_w-Kph2t7PT6U_Y3vKOfFZ4LUMFKaLiQi0MhND-zQNKBq1Sy1sjlkmXvBnMKGM8o_nGLshvDuWwmxcT5UYx_f8KRjugpIyRZwLolLMQrR9KL9qwAEI7_VUDmZSbK3WuMptDIb8VmItN7fF2c4DEXmYbkVroIW5DHFAYjZJaAgX7NkD

{'messages': [],
 'coordinator_agent': {'iteration': 1,
  'final_answer': False,
  'plan': [{'agent': 'warehouse_manager_agent',
    'task': 'Reserve 1x Lekaby Wireless Earbuds Bluetooth 5.3 (B0BRTRHQKX) and 1x HP Newest Notebook Laptop, 14" HD Touchscreen (B09XJJNCXZ) from the warehouse for the user. The items are already in the cart.'}],
  'next_agent': 'warehouse_manager_agent'},
 'answer': ''}

In [69]:
def evaluate_coordinator_delegation(run, example) -> bool:
    final_answer_match = run["coordinator_agent"]["final_answer"] == example["coordinator_agent"]["final_answer"]
    next_agent_match = run["coordinator_agent"]["next_agent"] == example["coordinator_agent"]["next_agent"]
    return final_answer_match and next_agent_match
    

In [70]:
evaluate_coordinator_delegation(output, reference_outputs[0])

True

In [71]:
def evaluate_coordinator_delegation_ls(run, example) -> bool:
    final_answer_match = run.outputs["coordinator_agent"]["final_answer"] == example.outputs["coordinator_agent"]["final_answer"]
    next_agent_match = run.outputs["coordinator_agent"]["next_agent"] == example.outputs["coordinator_agent"]["next_agent"]
    return final_answer_match and next_agent_match
    

In [72]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(State(
        messages=x["input"]["messages"],
        coordinator_agent=CoordinatorAgentProperties(
            iteration=0,
            final_answer=False,
            plan=[],
            next_agent=""
        ),
        answer=""
    )), #input of reference dataset iteraetion (injecting question into rag pipeline)
    data='coordinator-delegation-evaluation',
    evaluators=[
        evaluate_coordinator_delegation_ls
        # ragas_faithfulness,
        # ragas_relevancy,
    ],
    experiment_prefix='coordinator-delegation-evaluation',
    max_concurrency=10,
    num_repetitions=5
)

View the evaluation results for experiment: 'coordinator-delegation-evaluation-0d86280a' at:
https://smith.langchain.com/o/9ca9f230-7d5e-43a3-b1fe-2ad4d24d8dc9/datasets/3bce65b2-a574-4c1d-a848-5fb0b51c8376/compare?selectedSessions=a4c40349-19c0-4268-81a2-c2fa6fd38ff4




0it [00:00, ?it/s]